# 01 — Exploratory Data Analysis: Food-40 subset of Food-101

This notebook:
1. Loads the Food-101 dataset from Hugging Face and filters it to our 40 classes
2. Shows a 5×8 grid of sample images (one per class)
3. Plots the class distribution
4. Prints the train/val/test split sizes

Run from the project root (or let the first cell fix the path). The first run
downloads Food-101 (~5 GB) into `data/` — subsequent runs use the cache.

In [ ]:
# --- Setup: make the project root importable so `from src...` works ---
import sys
from pathlib import Path

# Notebook lives in notebooks/, so the project root is one level up
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import Config
from src.utils import set_seed
from src.data import FOOD40, load_food40_splits

# Seed everything so any sampling below is reproducible
set_seed(Config.SEED)

print(f"Device: {Config.DEVICE}")
print(f"Number of classes: {len(FOOD40)}")

In [ ]:
# --- Load the dataset and create the 70/15/15 split ---
# load_food40_splits() downloads/caches Food-101, filters to our 40 classes,
# and performs a stratified, seeded split (seed=42) so it is fully reproducible.
train_ds, val_ds, test_ds, label_map = load_food40_splits()

# label_map maps the original Food-101 label id -> our compact 0..39 id.
# Invert it so we can go from our id back to the original id when needed.
inv_label_map = {v: k for k, v in label_map.items()}

print("Split created successfully.")

In [ ]:
# --- Print split sizes ---
n_train, n_val, n_test = len(train_ds), len(val_ds), len(test_ds)
n_total = n_train + n_val + n_test

print(f"Train: {n_train:>6} images  ({n_train / n_total:.1%})")
print(f"Val:   {n_val:>6} images  ({n_val / n_total:.1%})")
print(f"Test:  {n_test:>6} images  ({n_test / n_total:.1%})")
print(f"Total: {n_total:>6} images across {len(FOOD40)} classes")

In [ ]:
# --- 5x8 grid: one sample image per class ---
# For each of the 40 classes, find the first training image with that label
# and display it. Titles use the human-readable class name.

# Build an index: our class id -> position of the first example in train_ds.
# We scan the label column once (fast, no image decoding happens here).
labels_col = train_ds["label"]  # original Food-101 ids
first_idx = {}
for pos, orig_label in enumerate(labels_col):
    cid = label_map[orig_label]
    if cid not in first_idx:
        first_idx[cid] = pos
    if len(first_idx) == len(FOOD40):
        break  # found one example for every class, stop scanning

# Plot the grid: 5 rows x 8 columns = 40 classes
fig, axes = plt.subplots(5, 8, figsize=(20, 13))
for cid, ax in enumerate(axes.flat):
    item = train_ds[first_idx[cid]]        # decodes just this one image
    ax.imshow(item["image"].convert("RGB"))
    ax.set_title(FOOD40[cid].replace("_", " "), fontsize=9)
    ax.axis("off")

fig.suptitle("One sample per class — Food-40 training set", fontsize=16)
fig.tight_layout()
plt.show()

In [ ]:
# --- Class distribution bar chart (training set) ---
# Count how many training images each class has. Food-101 is balanced at the
# source (1000 per class), so after a stratified split we expect near-equal
# bars — this plot is our visual proof, and it also justifies whether the
# optional class-weights extension is needed.
counts = pd.Series(labels_col).map(label_map).value_counts().sort_index()
counts.index = [FOOD40[i].replace("_", " ") for i in counts.index]

plt.figure(figsize=(16, 6))
sns.barplot(x=counts.index, y=counts.values, color="#4c72b0")
plt.xticks(rotation=75, ha="right", fontsize=9)
plt.ylabel("Training images")
plt.title("Class distribution — Food-40 training split")
plt.tight_layout()
plt.show()

# Print min/max as a quick imbalance check
print(f"Min class count: {counts.min()}  |  Max class count: {counts.max()}")

## Takeaways for the report (section 2.1)

- 40 classes selected with a coherent theme: dishes common in European nutrition apps
  (pasta, salads, grilled meats, sandwiches, soups, breakfast items, desserts).
- Fixed-seed (42), stratified 70/15/15 split → reproducible and class-balanced.
- Class distribution is near-uniform, so class weights are optional rather than necessary.